# Time is the Hardest Dimension

Time dimensions are special: they're naturally ordered, and they need an aggregation granularity (daily, weekly, monthly). This makes them the foundation for a family of transforms that are simple to express but non-trivial to implement in SQL.

This notebook demonstrates SLayer's time-related features with working code.

See also: [Expressions](../../concepts/expressions.md) | [Queries](../../concepts/queries.md)

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, datasets = ensure_jaffle_shop()

## Time Dimensions and Granularity

A time dimension truncates a date/timestamp field to a given granularity. SLayer supports `day`, `week`, `month`, `quarter`, and `year`.

Let's start with monthly revenue:

In [2]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "orders.ordered_at", "granularity": "month"}],
        "metrics": ["sum(orders.order_total)"],
        "order": [{"field": "orders.ordered_at", "direction": "asc"}],
        "limit": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>14}")
print("-" * 28)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    print(f"{month:<12} ${rev:>13,.2f}")

Month               Revenue
----------------------------
2023-11      $     6,105.41
2023-12      $    21,662.12
2024-01      $    25,295.56
2024-02      $    31,306.53
2024-03      $    41,006.82
2024-04      $    46,656.53
2024-05      $    51,172.88
2024-06      $    91,082.97
2024-07      $    95,861.30
2024-08      $   106,092.12
2024-09      $   100,498.25
2024-10      $   107,795.71


## Month-over-Month Change

The `change()` transform computes the difference from the previous period. It needs no extra arguments — the time dimension and granularity are already known from the query. Internally it uses a self-join CTE (not LAG), so it handles time gaps correctly: the previous period is matched on the calendar (e.g., the month before), not on the previous *row* in the result set. The very first period in the window still has no prior period to compare against, so its change is NULL (the first row below shows `-`); subsequent periods are gap-safe.

`change_pct()` gives the percentage change:

In [3]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "orders.ordered_at", "granularity": "month"}],
        "metrics": [
            "sum(orders.order_total)",
            {"metric": "change(sum(orders.order_total))", "alias": "mom_change"},
            {"metric": "change_pct(sum(orders.order_total))", "alias": "mom_pct"},
        ],
        "order": [{"field": "orders.ordered_at", "direction": "asc"}],
        "limit": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'MoM Change':>14} {'MoM %':>8}")
print("-" * 52)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["mom_change"]
    pct = row["mom_pct"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    pct_str = f"{pct:+.1%}" if pct is not None else "-"
    print(f"{month:<12} ${rev:>13,.2f} {chg_str:>14} {pct_str:>8}")

Month               Revenue     MoM Change    MoM %
----------------------------------------------------
2023-11      $     6,105.41              -        -
2023-12      $    21,662.12       $+15,557  +254.8%
2024-01      $    25,295.56        $+3,633   +16.8%
2024-02      $    31,306.53        $+6,011   +23.8%
2024-03      $    41,006.82        $+9,700   +31.0%
2024-04      $    46,656.53        $+5,650   +13.8%
2024-05      $    51,172.88        $+4,516    +9.7%
2024-06      $    91,082.97       $+39,910   +78.0%
2024-07      $    95,861.30        $+4,778    +5.2%
2024-08      $   106,092.12       $+10,231   +10.7%
2024-09      $   100,498.25        $-5,594    -5.3%
2024-10      $   107,795.71        $+7,297    +7.3%


## Year-over-Year with time_shift

`time_shift(metric, offset, granularity)` shifts the time dimension by the given offset before applying the query's granularity. This is implemented as a calendar-based self-join CTE — semantically simple, but the SQL is non-trivial.

For example, `time_shift(sum(orders.order_total), -1, 'year')` gives "same month, previous year":

In [4]:
import datetime as dt

# Use last full calendar year so the example stays in the demo's data window
# (the loader shifts dates so MAX(ordered_at) == today, with ~4 years of history).
last_year = dt.date.today().year - 1
date_range = [f"{last_year}-01-01", f"{last_year}-12-31"]

result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [
            {
                "field": "orders.ordered_at",
                "granularity": "month",
                "date_range": date_range,
            }
        ],
        "metrics": [
            "sum(orders.order_total)",
            {"metric": "time_shift(sum(orders.order_total), -1, 'year')", "alias": "prev_year"},
        ],
        "order": [{"field": "orders.ordered_at", "direction": "asc"}],
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'Prev Year':>14}")
print("-" * 42)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    prev = row["prev_year"]
    prev_str = f"${prev:>13,.2f}" if prev is not None else f"{'n/a':>14}"
    print(f"{month:<12} ${rev:>13,.2f} {prev_str}")

Month               Revenue      Prev Year
------------------------------------------
2025-01      $   169,658.45 $    25,295.56
2025-02      $   160,413.14 $    31,306.53
2025-03      $   187,221.94 $    41,006.82
2025-04      $   181,006.89 $    46,656.53
2025-05      $   189,947.37 $    51,172.88
2025-06      $   182,463.81 $    91,082.97
2025-07      $   200,077.85 $    95,861.30
2025-08      $   246,831.82 $   106,092.12
2025-09      $   203,101.39 $   100,498.25
2025-10      $   215,923.69 $   107,795.71
2025-11      $   225,577.11 $   114,670.96
2025-12      $   300,352.58 $   149,137.82


### time_shift vs lag

`time_shift` always uses a calendar-based self-join, with or without a granularity argument. Without one, `time_shift(x, -1)` shifts by one bucket of the query's own time granularity; with one (like `'year'` above), the shift can be any calendar amount — not just a multiple of the query granularity. Either way it's gap-safe: the previous value is matched on the actual calendar date, so a missing intermediate month produces NULL rather than the wrong value. The join also matches on all non-time dimensions, so grouped series (e.g. per-store) reset cleanly per group.

`lag(x, 1)` / `lead(x, 1)` use SQL's `LAG()`/`LEAD()` window functions instead — more efficient, but they blindly walk rows: if the result has a gap (e.g., a missing month), the previous row is the wrong period. Both approaches return NULL at the start of the window when there's no prior period in the data to match.

## The last() Transform

`last(metric)` returns the most recent time period's value, broadcast to every row. This **collapses the time dimension** — useful for comparing each period to the latest one, or for filtering.

For example, revenue by store alongside each store's most recent month's revenue:

In [5]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "orders.ordered_at", "granularity": "quarter"}],
        "metrics": [
            "sum(orders.order_total)",
            {"metric": "last(sum(orders.order_total))", "alias": "latest_quarter"},
        ],
        "dimensions": ["stores.name"],
        "order": [{"field": "orders.ordered_at", "direction": "desc"}],
        "limit": 10,
    }
)

print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'Latest Qtr':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["stores.name"]
    rev = row["orders.order_total_sum"]
    latest = row["latest_quarter"]
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} ${latest:>13,.2f}")

Quarter      Store                       Revenue     Latest Qtr
--------------------------------------------------------------
2026-10      San Francisco        $   113,425.25 $   113,425.25
2026-10      Chicago              $   126,050.14 $   126,050.14
2026-10      New Orleans          $    58,461.43 $    58,461.43
2026-10      Brooklyn             $   155,606.40 $   155,606.40
2026-10      Philadelphia         $    95,252.18 $    95,252.18
2026-07      Brooklyn             $   334,130.34 $   155,606.40
2026-07      Philadelphia         $   205,922.49 $    95,252.18
2026-07      New Orleans          $    88,042.09 $    58,461.43
2026-07      San Francisco        $   252,445.67 $   113,425.25
2026-07      Chicago              $   279,134.24 $   126,050.14


## Composing Transforms in Filters

Transforms can be composed and used in filters. For example, `last(change(sum(orders.order_total))) < 0` keeps only rows where the most recent period's change was negative — i.e., revenue declined in the latest period.

This is a powerful pattern: "show me all stores, but only those whose most recent quarter-over-quarter revenue went down":

In [6]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "orders.ordered_at", "granularity": "quarter"}],
        "metrics": [
            "sum(orders.order_total)",
            {"metric": "change(sum(orders.order_total))", "alias": "qoq_change"},
        ],
        "dimensions": ["stores.name"],
        "filter": ["last(change(sum(orders.order_total))) < 0"],
        "order": [{"field": "orders.ordered_at", "direction": "desc"}],
        "limit": 10,
    }
)

print("Stores where latest QoQ revenue declined:")
print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'QoQ Change':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["stores.name"]
    rev = row["orders.order_total_sum"]
    chg = row["qoq_change"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} {chg_str:>14}")

Stores where latest QoQ revenue declined:
Quarter      Store                       Revenue     QoQ Change
--------------------------------------------------------------
2026-10      San Francisco        $   113,425.25      $-139,020
2026-10      Chicago              $   126,050.14      $-153,084
2026-10      Philadelphia         $    95,252.18      $-110,670
2026-10      Brooklyn             $   155,606.40      $-178,524
2026-10      New Orleans          $    58,461.43       $-29,581
2026-07      Brooklyn             $   334,130.34       $-27,349
2026-07      New Orleans          $    88,042.09       $+62,822
2026-07      Chicago              $   279,134.24       $-12,481
2026-07      Philadelphia         $   205,922.49       $-19,563
2026-07      San Francisco        $   252,445.67        $-4,894


## Summary

Each of these concepts is simple and natural at the semantic level, but requires non-trivial SQL (CTEs, self-joins, window functions) to implement. SLayer offloads that effort.

| Transform | What it does | SQL implementation |
|-----------|-------------|-------------------|
| `change(x)` | Difference from previous period | Self-join CTE (gap-safe; first period is NULL) |
| `change_pct(x)` | Percentage change from previous period | Same as change, divided |
| `time_shift(x, -1, 'year')` | Same metric, shifted in time | Calendar-based self-join CTE |
| `lag(x)` / `lead(x)` | Previous/next row's value | SQL LAG/LEAD window (gap-blind) |
| `last(x)` | Most recent period's value, broadcast | Sub-query with FIRST_VALUE |
| `cumsum(x)` | Running total over time | SUM window function |

All transforms compose: `cumsum(change(x))`, `last(change(x))` in filters, etc.

See the [Expressions docs](../../concepts/expressions.md) for the full reference.